In [ ]:
# ══════════════════════════════════════════════════════════════════
# 🚨 NUCLEAR FIX for RAGAS ↔ langchain_community ChatVertexAI error
# ══════════════════════════════════════════════════════════════════
# Colab often ships an OLD ragas (0.1.x) that hard-imports a path
# removed from langchain_community 0.3+.  We force-remove it, then
# install a known-good matching set (ranges — resilient to version churn).
#
# 👉  RUN THIS CELL FIRST.
# 👉  When you see "✅ Installed" below, click:
#         Runtime → Restart runtime   (Ctrl+M .)
# 👉  THEN run the rest of the notebook from cell 2 downwards.
# ══════════════════════════════════════════════════════════════════

# 1. Uninstall the broken preinstalled versions (any that exist)
!pip uninstall -y -q ragas langchain langchain-core langchain-community langchain-groq langchain-text-splitters langchain-huggingface 2>/dev/null

# 2. Install a compatible matching set — RANGES, not exact pins,
#    so this cell doesn't break when new patch versions ship.
!pip install -q \
    "ragas>=0.2.10,<0.3" \
    "langchain>=0.3,<0.4" \
    "langchain-core>=0.3,<0.4" \
    "langchain-community>=0.3,<0.4" \
    "langchain-groq>=0.3,<1.0" \
    "langchain-text-splitters>=0.3,<0.4" \
    "langchain-huggingface>=0.1,<0.4" \
    "datasets>=3.0" \
    groq python-dotenv \
    sentence-transformers faiss-cpu rank_bm25 pymupdf \
    pandas matplotlib gradio

# 3. Verify installs succeeded (checks disk state — NOT cached imports)
import subprocess, json
def _v(pkg):
    r = subprocess.run(["pip", "show", pkg], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return "❌ NOT INSTALLED"

print("\n✅ Installed on disk:")
for p in ["ragas", "langchain", "langchain-core", "langchain-community",
          "langchain-groq", "langchain-huggingface", "datasets"]:
    print(f"   {p:<25} = {_v(p)}")

print("\n" + "═" * 60)
print("🔁 NEXT STEP (DO NOT SKIP):")
print("   Runtime → Restart runtime   (or press Ctrl+M .)")
print("   THEN run the notebook from cell 2 downwards.")
print("═" * 60)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 

# 📚 RAG Demo — Naive vs Advanced (with RAGAS Evaluation)

**What this notebook shows**
1. **Naive RAG** — PDF → chunk → embed → FAISS → LLM
2. **Advanced RAG** — Hybrid retrieval (FAISS + BM25) + RRF fusion + Cross-Encoder reranking
3. **RAGAS evaluation** — measure both pipelines on 4 quality metrics

**Stack:** Groq (Llama-3.1-8B + 3.3-70B) · LangChain · FAISS · BM25 · sentence-transformers · RAGAS

---

## ✅ Runs anywhere
Google Colab · Jupyter · JupyterLab · VS Code · Kaggle · local Python · Docker.
The setup cells auto-detect your environment.

## 🔧 Prerequisites

| Requirement | Details |
|---|---|
| **Python** | 3.9 or newer |
| **RAM** | 4 GB+ (embeddings + FAISS index) |
| **Disk** | ~500 MB free (HuggingFace models cache) |
| **GROQ_API_KEY** | Free at [console.groq.com/keys](https://console.groq.com/keys) |
| **Internet** | Needed to download the model + sample PDF on first run |

## 🔑 How to set your `GROQ_API_KEY` (pick one)

| Environment | How |
|---|---|
| **Google Colab** | Left sidebar 🔑 → Add secret named `GROQ_API_KEY` |
| **Jupyter / JupyterLab / VS Code** | Create a `.env` file with `GROQ_API_KEY=gsk_...` |
| **Local terminal (Linux/Mac)** | `export GROQ_API_KEY=gsk_...` |
| **Local terminal (Windows)** | `setx GROQ_API_KEY gsk_...` then restart shell |
| **Just running once** | The setup cell will prompt you securely |

## 🚀 Quick start

1. Run the install cell (cell 2). First time takes ~2 min.
2. Run the setup cell — it will load your key from the source above.
3. Run the rest top-to-bottom. The PDF is auto-downloaded if missing.

> ⚠️ **Free Groq tier** has a daily token limit (~100K TPD on 70B models). The RAGAS evaluation step is the heaviest — if you hit a 429 rate-limit error, wait ~15 min or switch the judge model to `llama-3.1-8b-instant`.


In [ ]:
# ── Install dependencies (works on Colab / Jupyter / VS Code / local) ──
# Note: On Google Colab you may see a harmless warning about google-colab
# requiring requests==2.32.4 — that's a notice, not a failure.
#
# ⚠️  Versions pinned to a known-good set to avoid the RAGAS ↔ langchain_community
#     'ChatVertexAI' ImportError. If you ever hit it, run pip -U on ragas + langchain*.
!pip install -q \
    groq gradio python-dotenv \
    "langchain>=0.3" "langchain-core>=0.3" "langchain-community>=0.3" \
    "langchain-groq>=0.2" "langchain-text-splitters>=0.3" "langchain-huggingface>=0.1" \
    sentence-transformers faiss-cpu rank_bm25 pymupdf \
    "ragas>=0.2.10" datasets pandas matplotlib \
    "requests>=2.32.4,<2.33"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 280.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360

In [ ]:
# Importing libraries
import gradio as gr
import os
import re
import numpy as np
from typing import List

# LangChain
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Groq
from groq import Groq


In [ ]:
# ══════════════════════════════════════════════════════════════════
# NAIVE RAG — Complete End-to-End
# Flow: PDF → Load → Chunk → Embed → Vector DB → Query → Retrieve → LLM → Answer
# ══════════════════════════════════════════════════════════════════
import os
from getpass import getpass

# ── Portable API key loader — works in Colab, Jupyter, VS Code, local Python ──
def load_groq_key():
    # 1. Already in environment? (CI/CD, Docker, exported in shell)
    if os.environ.get("GROQ_API_KEY"):
        return "environment"
    # 2. Google Colab secrets
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        return "colab-secrets"
    except (ImportError, ModuleNotFoundError):
        pass
    # 3. .env file (local dev with python-dotenv)
    try:
        from dotenv import load_dotenv
        if load_dotenv() and os.environ.get("GROQ_API_KEY"):
            return "dotenv"
    except ImportError:
        pass
    # 4. Interactive prompt (Jupyter / VS Code / terminal)
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ").strip()
    return "prompt"

source = load_groq_key()
print(f"✅ GROQ_API_KEY loaded from: {source}")


✅ GROQ_API_KEY loaded from: colab-secrets


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — Setup LLM client
# ─────────────────────────────────────────────────────────────────
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt: str) -> str:
    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return res.choices[0].message.content


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — READ PDF (auto-downloads if missing — works on any OS)
# ─────────────────────────────────────────────────────────────────
import os, urllib.request, tempfile

# Use /content if on Colab, else use OS-appropriate temp dir
if os.path.isdir("/content"):
    PDF_DIR = "/content"
else:
    PDF_DIR = os.path.join(tempfile.gettempdir(), "rag_demo")
    os.makedirs(PDF_DIR, exist_ok=True)

PDF_PATH = os.path.join(PDF_DIR, "Attention is all you need.pdf")
if not os.path.exists(PDF_PATH):
    print(f"Downloading PDF to {PDF_PATH} ...")
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762.pdf", PDF_PATH)

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(f" Step 1 — Loaded {len(pages)} pages from PDF")


 Step 1 — Loaded 15 pages from PDF


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — CHUNK
# ─────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
docs = splitter.split_documents(pages)

# Add metadata for citations
for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i
    d.metadata["page"] = d.metadata.get("page", 0) + 1

print(f" Step 2 — Split into {len(docs)} chunks")



 Step 2 — Split into 52 chunks


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CHUNKING TRADE-OFF DEMO
# Same paragraph → 3 different chunk_size settings → see the impact.
# ─────────────────────────────────────────────────────────────────
sample_text = pages[1].page_content  # page 2 of the paper (has real prose)

for size, overlap in [(200, 20), (1000, 200), (3000, 300)]:
    demo_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    demo_chunks = demo_splitter.split_text(sample_text)
    print(f"\n▶ chunk_size={size:<4}  overlap={overlap:<3}  →  {len(demo_chunks):>2} chunks")
    print(f"   first chunk (first 120 chars): {demo_chunks[0][:120]!r}…")

print("\n📏 Rule of thumb: 500–1000 tokens with 10–20% overlap works for most PDFs.")
print("   • Too SMALL → context lost across boundaries → recall ↓")
print("   • Too BIG   → signal diluted, noisy retrieval → precision ↓")



▶ chunk_size=200   overlap=20   →  27 chunks
   first chunk (first 120 chars): '1\nIntroduction\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks'…

▶ chunk_size=1000  overlap=200  →   6 chunks
   first chunk (first 120 chars): '1\nIntroduction\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks\nin particu'…

▶ chunk_size=3000  overlap=300  →   2 chunks
   first chunk (first 120 chars): '1\nIntroduction\nRecurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks\nin particu'…

📏 Rule of thumb: 500–1000 tokens with 10–20% overlap works for most PDFs.
   • Too SMALL → context lost across boundaries → recall ↓
   • Too BIG   → signal diluted, noisy retrieval → precision ↓


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — EMBED + STORE in Vector DB (FAISS)
# ─────────────────────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(docs, embeddings)
print(f" Step 3 — Stored {len(docs)} chunks in FAISS vector DB")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Step 3 — Stored 52 chunks in FAISS vector DB


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — NAIVE RAG (retrieve from vector DB → LLM → answer)
# ─────────────────────────────────────────────────────────────────
NAIVE_PROMPT = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: "Not found in document".
Cite pages like [Page X].

Context:
{context}

Question: {question}

Answer:"""

def naive_rag(query: str, k: int = 5) -> dict:
    # 4a. RETRIEVE top-k similar chunks from vector DB
    retrieved = vectorstore.similarity_search(query, k=k)

    # 4b. BUILD context string (with page numbers for citations)
    context_blocks = []
    for d in retrieved:
        page = d.metadata.get("page", "?")
        context_blocks.append(f"[Page {page}]\n{d.page_content}")
    context = "\n\n---\n\n".join(context_blocks)

    # 4c. ASK the LLM
    prompt = NAIVE_PROMPT.format(context=context, question=query)
    answer = call_llm(prompt)

    return {
        "answer": answer,
        "retrieved_chunks": retrieved,
        "pages_used": sorted({d.metadata.get("page", "?") for d in retrieved}),
    }


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — DEMO
# ─────────────────────────────────────────────────────────────────
queries = [
    "What is the main idea of this paper?",
    "What is the Transformer architecture?",
    "What datasets were used in the experiments?",
    "Who are the authors of this paper?",
]

for q in queries:
    print("\n" + "═" * 70)
    print(f" Question: {q}")
    print("═" * 70)
    out = naive_rag(q)
    print(f" Pages used: {out['pages_used']}")
    print(f"\n Answer:\n{out['answer']}")


══════════════════════════════════════════════════════════════════════
 Question: What is the main idea of this paper?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 12, 13, 14, 15]

 Answer:
Not found in document.

══════════════════════════════════════════════════════════════════════
 Question: What is the Transformer architecture?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 2, 3, 5, 8]

 Answer:
The Transformer architecture follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, as shown in Figure 1 [Page 3].

══════════════════════════════════════════════════════════════════════
 Question: What datasets were used in the experiments?
══════════════════════════════════════════════════════════════════════
 Pages used: [4, 7, 9]

 Answer:
Not found in document.

═══════════════════════════════════════════════════════════════

---
### 🚨 Naive RAG hit a wall — let's diagnose

Notice how *some* answers looked great, but the **"Who are the authors?"** query returned **"Not found in document"** — even though the authors are literally on page 1.

**Why?** MiniLM embeddings encode **meaning**, not **exact tokens**.
Proper nouns ("Vaswani", "Shazeer"…) don't have strong semantic neighbors in a paper about attention — so the FAISS top-k misses them entirely.

Run the cell below to *see* the miss, then we'll fix it in Part 2 with BM25 + reranking.


In [ ]:
# ─────────────────────────────────────────────────────────────────
# 🔬 DIAGNOSTIC — why does Naive RAG miss the authors?
# Look at what FAISS actually retrieved for that query.
# ─────────────────────────────────────────────────────────────────
diag_query = "Who are the authors of this paper?"

print(f"❓ Query: {diag_query}\n")
print("🔵 What FAISS (dense / semantic) returned as top-5:")
print("─" * 70)
faiss_hits = vectorstore.similarity_search(diag_query, k=5)
for i, d in enumerate(faiss_hits, 1):
    snippet = d.page_content.replace("\n", " ")[:110]
    print(f"  {i}. [Page {d.metadata.get('page','?')}]  {snippet}…")

# Does the authors chunk appear anywhere in the top 5?
authors_in_top5 = any("Vaswani" in d.page_content or "Shazeer" in d.page_content
                      for d in faiss_hits)
print("\n" + "─" * 70)
print(f"👀 Is the authors chunk in the top-5?  →  {'✅ YES' if authors_in_top5 else '❌ NO — that is why the answer was Not found'}")
print("\n💡 Take-away: semantic search alone cannot retrieve proper nouns")
print("   that don't have a strong meaning-vector neighborhood.")
print("   → Fix in Part 2:  add BM25 (keyword search) so exact tokens are catchable.")


❓ Query: Who are the authors of this paper?

🔵 What FAISS (dense / semantic) returned as top-5:
──────────────────────────────────────────────────────────────────────
  1. [Page 10]  comments, corrections and inspiration. References [1] Jimmy Lei Ba, Jamie Ryan Kiros, and Geoffrey E Hinton. L…
  2. [Page 12]  [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building a large annotated corpus of e…
  3. [Page 11]  across languages. In Proceedings of the 2009 Conference on Empirical Methods in Natural Language Processing, p…
  4. [Page 12]  [37] Vinyals & Kaiser, Koo, Petrov, Sutskever, and Hinton. Grammar as a foreign language. In Advances in Neura…
  5. [Page 11]  2017. [19] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured attention networks. In Intern…

──────────────────────────────────────────────────────────────────────
👀 Is the authors chunk in the top-5?  →  ✅ YES

💡 Take-away: semantic search alone cannot retrieve proper nouns
   that d

---
## 🚀 Part 2 — Advanced RAG

Adds three quality improvements on top of the naive pipeline:
1. **Hybrid retrieval** — FAISS (semantic) + BM25 (keyword)
2. **RRF fusion** — combines both ranked lists
3. **Cross-Encoder reranker** — precision filter on the top-N


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Advanced RAG — Hybrid retrieval + RRF + Cross-Encoder reranking
# Flow: PDF → Load → Chunk → [FAISS + BM25] → RRF → Rerank → LLM
# ══════════════════════════════════════════════════════════════════
# (All needed libraries were already imported at the top of the notebook.
#  GROQ_API_KEY is already loaded into os.environ by the portable loader.)
import os


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 0 — LLM client (reuses the same key loaded earlier)
# ─────────────────────────────────────────────────────────────────
client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_llm(prompt: str) -> str:
    res = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return res.choices[0].message.content


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 1 — LOAD PDF (reuses PDF_PATH from the Naive RAG section)
# ─────────────────────────────────────────────────────────────────
loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(f" Step 1 — Loaded {len(pages)} pages")


 Step 1 — Loaded 15 pages


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 2 — CHUNK
# ─────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = splitter.split_documents(pages)
for i, d in enumerate(docs):
    d.metadata["chunk_id"] = i
    d.metadata["page"] = d.metadata.get("page", 0) + 1
print(f"  Step 2 — Split into {len(docs)} chunks")


  Step 2 — Split into 52 chunks


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 3 — BUILD 3 INDEXES (Dense + Sparse + Reranker)
# ─────────────────────────────────────────────────────────────────
# 3a. Dense vector index (FAISS)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)

# 3b. Sparse keyword index (BM25)
def preprocess(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return text.split()

bm25 = BM25Okapi([preprocess(d.page_content) for d in docs])

# 3c. Cross-Encoder reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")

print(" Step 3 — FAISS + BM25 + Cross-Encoder ready")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

 Step 3 — FAISS + BM25 + Cross-Encoder ready


In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 4 — RRF FUSION (combine ranked lists)
# ─────────────────────────────────────────────────────────────────
def rrf(result_lists, k: int = 60):
    """Reciprocal Rank Fusion: score = Σ 1/(k + rank)"""
    scores = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = hash(doc.page_content)
            scores.setdefault(key, {"doc": doc, "score": 0.0})
            scores[key]["score"] += 1.0 / (rank + k + 1)
    return sorted(scores.values(), key=lambda x: x["score"], reverse=True)



In [ ]:
# ─────────────────────────────────────────────────────────────────
# STEP 5 — ADVANCED RAG PIPELINE
# ─────────────────────────────────────────────────────────────────
ADVANCED_PROMPT = """You are a precise assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say: "Not found in document".
Cite pages like [Page X].

Context:
{context}

Question: {question}

Answer:"""

def advanced_rag(query: str, top_k: int = 5, fetch_k: int = 20) -> dict:
    # 5a. DENSE retrieval (semantic)
    sem_docs = vectorstore.similarity_search(query, k=fetch_k)

    # 5b. SPARSE retrieval (BM25 keyword)
    bm_scores = bm25.get_scores(preprocess(query))
    bm_idx = np.argsort(bm_scores)[::-1][:fetch_k]
    kw_docs = [docs[i] for i in bm_idx]

    # 5c. RRF FUSION (merge both ranked lists)
    fused = [x["doc"] for x in rrf([sem_docs, kw_docs])]

    # 5d. CROSS-ENCODER RERANK (precision filter)
    pairs = [(query, d.page_content) for d in fused]
    ce_scores = reranker.predict(pairs)
    ranked = sorted(zip(fused, ce_scores), key=lambda x: x[1], reverse=True)
    top_docs = [d for d, _ in ranked[:top_k]]

    # 5e. BUILD context with citations
    context = "\n\n---\n\n".join(
        f"[Page {d.metadata.get('page','?')}]\n{d.page_content}" for d in top_docs
    )

    # 5f. CALL LLM
    answer = call_llm(ADVANCED_PROMPT.format(context=context, question=query))

    return {
        "answer": answer,
        "retrieved_chunks": top_docs,
        "pages_used": sorted({d.metadata.get("page", "?") for d in top_docs}),
    }


In [ ]:

# ─────────────────────────────────────────────────────────────────
# STEP 6 — DEMO (same 4 questions as Naive RAG)
# ─────────────────────────────────────────────────────────────────
queries = [
    "What is the main idea of this paper?",
    "What is the Transformer architecture?",
    "What datasets were used in the experiments?",
    "Who are the authors of this paper?",
]

for q in queries:
    print("\n" + "═" * 70)
    print(f" Question: {q}")
    print("═" * 70)
    out = advanced_rag(q)
    print(f" Pages used: {out['pages_used']}")
    print(f"\n Answer:\n{out['answer']}")


══════════════════════════════════════════════════════════════════════
 Question: What is the main idea of this paper?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 2, 7, 12]

 Answer:
The main idea of this paper is to propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions.

══════════════════════════════════════════════════════════════════════
 Question: What is the Transformer architecture?
══════════════════════════════════════════════════════════════════════
 Pages used: [1, 2, 3, 9]

 Answer:
The Transformer architecture follows an overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, as shown in Figure 1 [Page 3].

══════════════════════════════════════════════════════════════════════
 Question: What datasets were used in the experiments?
══════════════════════════════════════════════

---
### 🎯 The Reveal — what each retriever *actually* pulled

Same query as the naive diagnostic ("Who are the authors…"). Watch:
- **Dense (FAISS)** — still misses the authors chunk (embeddings don't help for proper nouns)
- **Sparse (BM25)** — finds it *immediately* because "authors" and last-name tokens are literal matches
- **Advanced (Hybrid + Rerank)** — union of the two, with the most relevant chunk lifted to rank 1 by the cross-encoder

This one cell explains *why* Advanced RAG's RAGAS scores jump so much in Part 3.


In [ ]:
q = "Who are the authors of this paper?"

print(" DENSE (FAISS) top 5:")
for d in vectorstore.similarity_search(q, k=5):
    print(f"   Page {d.metadata.get('page')}: {d.page_content[:80]}…")

print("\n SPARSE (BM25) top 5:")
bm_idx = np.argsort(bm25.get_scores(preprocess(q)))[::-1][:5]
for i in bm_idx:
    print(f"   Page {docs[i].metadata.get('page')}: {docs[i].page_content[:80]}…")

print("\n ADVANCED (Hybrid + Rerank) top 5:")
out = advanced_rag(q)
for d in out["retrieved_chunks"]:
    print(f"   Page {d.metadata.get('page')}: {d.page_content[:80]}…")

 DENSE (FAISS) top 5:
   Page 10: comments, corrections and inspiration.
References
[1] Jimmy Lei Ba, Jamie Ryan K…
   Page 12: [25] Mitchell P Marcus, Mary Ann Marcinkiewicz, and Beatrice Santorini. Building…
   Page 11: across languages. In Proceedings of the 2009 Conference on Empirical Methods in …
   Page 12: [37] Vinyals & Kaiser, Koo, Petrov, Sutskever, and Hinton. Grammar as a foreign …
   Page 11: 2017.
[19] Yoon Kim, Carl Denton, Luong Hoang, and Alexander M. Rush. Structured…

 SPARSE (BM25) top 5:
   Page 7: and semantic structure of the sentences.
5
Training
This section describes the t…
   Page 7: We trained our models on one machine with 8 NVIDIA P100 GPUs. For our base model…
   Page 1: Provided proper attribution is provided, Google hereby grants permission to
repr…
   Page 15: The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
wha…
   Page 6: tokens in the sequence. To this end, we add "positional encodings" to the input …

 ADVANCED (Hybrid 

---
## 📊 Part 3 — RAGAS Evaluation

Compare both pipelines on 4 metrics: **faithfulness, answer relevancy, context precision, context recall**.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 1 — Golden test set + run both pipelines
# (RAGAS needs {question, answer, contexts, ground_truth} per row)
# ══════════════════════════════════════════════════════════════════
test_set = [
    {
        "question": "What is the main idea of this paper?",
        "ground_truth": (
            "The paper proposes the Transformer, a new network architecture "
            "based solely on attention mechanisms, dispensing entirely with "
            "recurrence and convolutions. It achieves superior translation "
            "quality while being more parallelizable and faster to train."
        ),
    },
    {
        "question": "What is the Transformer architecture?",
        "ground_truth": (
            "The Transformer follows an encoder-decoder structure using stacked "
            "self-attention and point-wise fully connected layers for both the "
            "encoder and decoder, as shown in Figure 1."
        ),
    },
    {
        "question": "What datasets were used in the experiments?",
        "ground_truth": (
            "WMT 2014 English-German (~4.5M sentence pairs), "
            "WMT 2014 English-French (36M sentences), "
            "Wall Street Journal portion of the Penn Treebank (~40K sentences), "
            "and high-confidence + BerkeleyParser corpora (~17M sentences)."
        ),
    },
    {
        "question": "Who are the authors of this paper?",
        "ground_truth": (
            "Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, "
            "Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin."
        ),
    },
]

def run_pipeline(rag_fn, testset, label):
    print(f"\n▶ Running {label}...")
    rows = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
    for i, item in enumerate(testset, 1):
        print(f"   [{i}/{len(testset)}] {item['question'][:50]}…")
        out = rag_fn(item["question"])
        rows["question"].append(item["question"])
        rows["answer"].append(out["answer"])
        rows["contexts"].append([d.page_content for d in out["retrieved_chunks"]])
        rows["ground_truth"].append(item["ground_truth"])
    return rows

naive_rows = run_pipeline(naive_rag, test_set, "Naive RAG")
adv_rows   = run_pipeline(advanced_rag, test_set, "Advanced RAG")
print("\n✅ Both pipelines completed")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 2 — Configure the RAGAS judge (Groq Llama + MiniLM embeddings)
#
# 💡 Judge model choice:
#    • llama-3.3-70b-versatile  → BETTER JSON output → fewer NaN scores  ← current default
#    • llama-3.1-8b-instant     → faster · more free quota, but noisier judge (more NaN rows)
#
# Hardened against the common NaN causes on the Groq free tier:
#    • max_retries=5      → auto-retry on 429 / transient errors
#    • timeout=60         → per-call safety net
#    • RunConfig(max_workers=1) → serial judge calls → no RPM-burst 429s
#    • nltk punkt primed → some RAGAS metrics need sentence-tokenisation
# ══════════════════════════════════════════════════════════════════
import os
from datasets import Dataset
from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    faithfulness, answer_relevancy,
    context_precision, context_recall,
)
try:
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
except ImportError:  # ragas >= 0.2 alt layout
    from ragas.llms.base import LangchainLLMWrapper
    from ragas.embeddings.base import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq

# Prime NLTK sentence tokenisers (some RAGAS metrics need them → silent NaN if missing)
try:
    import nltk
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

JUDGE_MODEL = "llama-3.3-70b-versatile"   # ← stronger judge → cleaner JSON → fewer NaN
# JUDGE_MODEL = "llama-3.1-8b-instant"    # ← swap here if 70B hits daily 100K TPD cap

# ChatGroq's kwarg name changed across versions — try both, and always request retries
try:
    chat = ChatGroq(
        model=JUDGE_MODEL, temperature=0,
        api_key=os.environ["GROQ_API_KEY"],
        max_retries=5, timeout=60,
    )
except TypeError:
    chat = ChatGroq(
        model=JUDGE_MODEL, temperature=0,
        groq_api_key=os.environ["GROQ_API_KEY"],
        max_retries=5, timeout=60,
    )

judge_llm        = LangchainLLMWrapper(chat)
judge_embeddings = LangchainEmbeddingsWrapper(embeddings)   # reuse MiniLM

METRICS = [faithfulness, answer_relevancy, context_precision, context_recall]

# Serial run config → prevents 429 bursts on Groq free tier → fewer NaN rows
RAGAS_RUN_CONFIG = RunConfig(max_workers=1, timeout=180, max_retries=5)

print(f"✅ RAGAS configured — judge: {JUDGE_MODEL} · embeddings: MiniLM")
print(f"   max_retries=5 · timeout=60 · serial worker → resilient to free-tier rate limits")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 3 — Run RAGAS scoring on both pipelines
# ══════════════════════════════════════════════════════════════════
import pandas as pd

def evaluate_pipeline(rows, label):
    print(f"\n  Evaluating {label} (4 metrics × {len(rows['question'])} questions)…")
    ds = Dataset.from_dict(rows)
    result = evaluate(
        ds,
        metrics=METRICS,
        llm=judge_llm,
        embeddings=judge_embeddings,
        run_config=RAGAS_RUN_CONFIG,        # ← serial + retries → fewer NaN
    )
    df = result.to_pandas()
    df.insert(0, "pipeline", label)
    return df

df_naive = evaluate_pipeline(naive_rows, "Naive")
df_adv   = evaluate_pipeline(adv_rows,   "Advanced")
df_all = pd.concat([df_naive, df_adv], ignore_index=True)
print("\n✅ Evaluation complete")
df_all


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 4 — 🏆 RAGAS REPORT  (NaN-safe · shows sample sizes · fair winner)
# ══════════════════════════════════════════════════════════════════
# Any single failed judge call → NaN for that row × metric. We handle that
# by (a) letting pandas .mean() skip NaN, (b) reporting the n rows that
# actually scored, and (c) choosing the winner only over metrics that BOTH
# pipelines managed to score, so one all-NaN column can't crown the wrong side.
# ══════════════════════════════════════════════════════════════════
import numpy as np

metric_cols = ["faithfulness", "context_precision", "context_recall", "answer_relevancy"]
pretty = {
    "faithfulness":      "Faithfulness",
    "context_precision": "Context Precision",
    "context_recall":    "Context Recall",
    "answer_relevancy":  "Answer Relevancy",
}
labels = {"Naive": "Naive RAG", "Advanced": "Advanced RAG"}

# Only score columns that RAGAS actually produced (guards against version drift)
present = [m for m in metric_cols if m in df_all.columns]

summary  = df_all.groupby("pipeline")[present].mean()      # .mean() already skips NaN
sample_n = df_all.groupby("pipeline")[present].count()      # non-NaN rows per metric
total_n  = df_all.groupby("pipeline").size()

print("=" * 40)
print("RAGAS REPORT")
print("=" * 40)

for pipe in ["Naive", "Advanced"]:
    if pipe not in summary.index:
        continue
    n_total = int(total_n.loc[pipe])
    print(f"\n{labels[pipe]}  ({n_total} questions)\n")
    for m in present:
        val   = summary.loc[pipe, m]
        n_ok  = int(sample_n.loc[pipe, m])
        if n_ok == 0 or np.isnan(val):
            print(f"{pretty[m]:<18} :  (no data — all {n_total} rows returned NaN)")
        else:
            tag = "" if n_ok == n_total else f"  [{n_ok}/{n_total} rows scored]"
            print(f"{pretty[m]:<18} : {val:.2f}{tag}")

# ── Winner: only compare metrics where BOTH pipelines have a real number ──
common_metrics = [
    m for m in present
    if "Naive"    in summary.index and not np.isnan(summary.loc["Naive", m])
    and "Advanced" in summary.index and not np.isnan(summary.loc["Advanced", m])
]

print("\n" + "-" * 40)
if not common_metrics:
    print("Winner :  (indeterminate — no metric has scores for both pipelines)")
elif "Naive" not in summary.index or "Advanced" not in summary.index:
    print("Winner :  (only one pipeline was evaluated)")
else:
    avg_naive = summary.loc["Naive", common_metrics].mean()
    avg_adv   = summary.loc["Advanced", common_metrics].mean()
    if   avg_adv   > avg_naive: winner = "Advanced RAG"
    elif avg_naive > avg_adv:   winner = "Naive RAG"
    else:                       winner = "Tie"

    metrics_used = ", ".join(pretty[m] for m in common_metrics)
    print(f"Winner (mean of {len(common_metrics)} shared metrics: {metrics_used}):")
    print(f"   {winner}   [ Advanced {avg_adv:.2f}  vs  Naive {avg_naive:.2f} ]")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 5 — Visual comparison (bar chart)
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
summary.T.plot(
    kind="bar", ax=ax, rot=15,
    color=["#C73E1D", "#2E86AB"],
    edgecolor="black", width=0.7,
)
ax.set_title("Naive vs Advanced RAG — RAGAS Quality Metrics",
             fontsize=14, fontweight="bold")
ax.set_ylabel("Score (0 = bad, 1 = perfect)")
ax.set_ylim(0, 1.05)
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5,
           label="Production threshold")
ax.legend(title="Pipeline", loc="lower right")
ax.grid(axis="y", alpha=0.3)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", padding=3, fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 6 — 🔬 NaN DIAGNOSTIC — which rows / metrics silently failed?
# ──────────────────────────────────────────────────────────────────
# Run this ONLY if the RAGAS REPORT above shows "(no data)" or
# "[n/4 rows scored]" for any metric. It prints per-row scores,
# NaN counts per metric, and flags any generated answer that is
# too short (or "Not found in document") for the judge to score.
# ══════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd

def show_scores(df, label):
    metric_only = [c for c in ["faithfulness", "context_precision",
                               "context_recall", "answer_relevancy"]
                   if c in df.columns]
    q_col = next((c for c in ["question", "user_input"] if c in df.columns), None)
    display_cols = ([q_col] if q_col else []) + metric_only

    print(f"\n▶ {label} — per-row scores:")
    with pd.option_context("display.max_colwidth", 55, "display.width", 200):
        print(df[display_cols].to_string(index=False))

    nan_counts = df[metric_only].isna().sum()
    print(f"\n  NaN counts (out of {len(df)} rows):")
    for m, n in nan_counts.items():
        mark = "  ⚠️" if n > 0 else "  ✅"
        print(f"    {m:<20}: {n}{mark}")

show_scores(df_naive, "NAIVE RAG")
show_scores(df_adv,   "ADVANCED RAG")

# ── Flag answers likely to break RAGAS ───────────────────────────
print("\n" + "═" * 70)
print(" Answers likely to break RAGAS  (< 40 chars OR contain 'not found')")
print("═" * 70)
found_any = False
for label, rows in [("Naive", naive_rows), ("Advanced", adv_rows)]:
    for q, a in zip(rows["question"], rows["answer"]):
        if len(a.strip()) < 40 or "not found" in a.lower():
            found_any = True
            print(f"  ⚠️  [{label:<8}] {q[:55]!r}")
            print(f"         ↳ answer: {a[:100]!r}\n")
if not found_any:
    print("  ✅ All answers look scoreable.")

# ── Root-cause quick reference ────────────────────────────────────
print("\n" + "═" * 70)
print(" Why NaN happens — in order of likelihood")
print("═" * 70)
print("  1. Answer was 'Not found in document'  → Faithfulness has no claims to verify → NaN")
print("     • Fix: expected — that row honestly cannot be scored on generation metrics")
print("  2. Groq 429 during a burst              → row silently dropped")
print("     • Fix: STEP 2 now uses max_retries=5 + RunConfig(max_workers=1)")
print("  3. 8B judge returned malformed JSON     → RAGAS parser rejected the response")
print("     • Fix: STEP 2 now defaults to the stronger llama-3.3-70b-versatile judge")
print("\n→ If you STILL see NaNs after these fixes, re-run STEP 3 once (transient 429).")


---
## 🛡️ STEP 7 — Faithfulness = Decompose + Verify (opened up on our Transformer RAG)

RAGAS's `faithfulness` metric in STEP 4 is not magic — it's a **two-pass LLM-as-Judge**
that anyone can implement in ~50 lines. Below we open the hood and run *the same
algorithm* on an answer produced by our `advanced_rag` on the Attention paper — so
you can see WHY the score comes out the way it does, and prove the mechanism actually
catches hallucinations.

### 🧠 The algorithm (mirrors the workshop slide)

```
   ①  Generated answer                   ②  Atomic claims                ③  Verify vs retrieved context
   ─────────────────────────             ──────────────────────           ─────────────────────────────
   "The Transformer follows an           A: encoder-decoder structure  →  CTX chunk with "encoder-decoder" ✅
    encoder-decoder structure        →   B: uses self-attention        →  CTX chunk with "self-attention" ✅
    with self-attention and              C: uses ReLU activations      →  CTX has no "ReLU" mention       ❌  ← hallucination caught
    ReLU activations."                                                     (fabricated claim)
                                     judge pass 1: DECOMPOSE           judge pass 2: ENTAILED? YES / NO

               Faithfulness  =  supported_claims / total_claims  =  2 / 3  =  0.67
```

**Key insight:** the model is asked *many small binary questions* ("is claim X in the
context?") instead of *one big fuzzy question* ("is this whole answer faithful?").
Small binary decisions are far more stable than a single 0-10 rating — that is the
whole reason this metric works.

### What the next two cells do

1. **STEP 7b** — Loads a helper `faithfulness_decompose_verify(answer, context)`
   that reuses the same Groq client + 70B judge we already configured in STEP 2.
2. **STEP 7c** — Runs it twice on the same Attention-paper question:
   - **Case A:** the real `advanced_rag` answer → should score high
   - **Case B:** we deliberately inject a fabricated claim ("*invented at Stanford in 2010*")
     → the verify step should catch it, dragging the score down
   - Cross-checks our manual score against RAGAS's own `faithfulness` from `df_adv`


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 7b — HELPER: manual Faithfulness = Decompose + Verify
# ──────────────────────────────────────────────────────────────────
# This is the ~50-line version of what RAGAS `faithfulness` does
# internally. We reuse the same Groq client that's already loaded
# (`client`) and the same 70B judge model (`JUDGE_MODEL`) so results
# are directly comparable to STEP 4's RAGAS report.
#
# Two LLM passes per answer:
#   Pass 1 — DECOMPOSE → JSON array of atomic factual claims
#   Pass 2 — VERIFY   → for each claim, YES / NO against the context
#
#   Faithfulness = supported_claims / total_claims
# ══════════════════════════════════════════════════════════════════
import json, re

def _judge_call(prompt: str) -> str:
    """Call the same 70B judge model RAGAS is using in STEP 3."""
    res = client.chat.completions.create(
        model=JUDGE_MODEL,          # llama-3.3-70b-versatile from STEP 2
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return res.choices[0].message.content

# ── Pass 1: DECOMPOSE ────────────────────────────────────────────
DECOMPOSE_PROMPT = """Extract every distinct FACTUAL claim from the answer below.
Rules:
  • Each claim must be a single, atomic, verifiable statement.
  • Split compound sentences into separate claims.
  • Return ONLY a JSON array of strings. No prose, no markdown fences.

Answer:
\"\"\"{answer}\"\"\""""

def decompose(answer: str) -> list[str]:
    raw = _judge_call(DECOMPOSE_PROMPT.format(answer=answer))
    # Robust JSON parsing (LLMs sometimes add ```json fences)
    m = re.search(r"\[[\s\S]*\]", raw)
    if not m:
        return [ln.lstrip("-•* \t") for ln in raw.splitlines() if ln.strip()]
    try:
        return [str(c).strip() for c in json.loads(m.group(0)) if str(c).strip()]
    except json.JSONDecodeError:
        return [ln.lstrip("-•* \t") for ln in raw.splitlines() if ln.strip()]

# ── Pass 2: VERIFY each claim against the retrieved context ──────
VERIFY_PROMPT = """Given ONLY the context below, is the claim entailed?
Reply with exactly ONE word: YES or NO.

Context:
\"\"\"{context}\"\"\"

Claim: {claim}"""

def verify(claim: str, context: str) -> bool:
    verdict = _judge_call(VERIFY_PROMPT.format(context=context, claim=claim))
    return verdict.strip().upper().startswith("YES")

# ── The full metric — matches RAGAS `faithfulness` shape ─────────
def faithfulness_decompose_verify(answer: str, context: str, verbose: bool = True):
    """Returns (score, list of (claim, supported_bool)) for one row."""
    if not answer or "not found" in answer.lower() or len(answer.strip()) < 20:
        if verbose:
            print("  ⚠️  Answer too short / refusal → no claims to verify → NaN")
        return float("nan"), []

    claims = decompose(answer)
    verdicts = [(c, verify(c, context)) for c in claims]
    supported = sum(1 for _, ok in verdicts if ok)
    score = supported / max(1, len(verdicts))

    if verbose:
        print("\n  ┌─────┬───────────────────────────────────────────────────┬───────────┐")
        print(  "  │ #   │ Atomic claim                                      │ Verdict   │")
        print(  "  ├─────┼───────────────────────────────────────────────────┼───────────┤")
        for i, (c, ok) in enumerate(verdicts, 1):
            mark = "✅ SUPPORTED" if ok else "❌ NOT IN CTX"
            print(f"  │ {i:<3} │ {c[:49]:<49} │ {mark:<9} │")
        print(  "  └─────┴───────────────────────────────────────────────────┴───────────┘")
        print(f"\n  Faithfulness  =  {supported} supported  /  {len(verdicts)} total  =  {score:.2f}")
    return score, verdicts

print("✅ Decompose+Verify helper loaded — judge:", JUDGE_MODEL)


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 7c — RUN Decompose + Verify on our Attention-paper RAG
# ──────────────────────────────────────────────────────────────────
# We run TWO cases on the same question so you can SEE the mechanism
# catch a hallucination:
#
#   Case A — REAL answer produced by advanced_rag (should score HIGH)
#   Case B — SAME question, but we inject a fake fabricated claim
#            ("…and it was invented at Stanford University")
#            → the verify step should flag it as UNSUPPORTED → score drops
#
# Finally we cross-check our score against RAGAS's own `faithfulness`
# score from df_adv for the same question — they should agree closely.
# ══════════════════════════════════════════════════════════════════
import pandas as pd

# ── Pick a substantive multi-claim question from our test set ────
target_q = "What is the Transformer architecture?"
idx = adv_rows["question"].index(target_q)

real_answer = adv_rows["answer"][idx]
retrieved   = adv_rows["contexts"][idx]
context_str = "\n\n---\n\n".join(retrieved)

# ── Case A: real Advanced RAG answer ─────────────────────────────
print("=" * 74)
print(" CASE A — REAL answer from advanced_rag")
print("=" * 74)
print(f"❓ Question: {target_q}\n")
print(f"💬 Answer:\n{real_answer[:400]}{'…' if len(real_answer) > 400 else ''}")
score_A, verdicts_A = faithfulness_decompose_verify(real_answer, context_str)

# ── Case B: same question, fabricated claim injected ─────────────
fabricated_answer = (
    real_answer.rstrip(".") +
    ". The Transformer was originally invented at Stanford University "
    "in 2010 and later adopted by Google."
)
print("\n" + "=" * 74)
print(" CASE B — SAME answer + a FABRICATED tail (Stanford / 2010)")
print("=" * 74)
print(f"💬 Answer (last 200 chars):\n…{fabricated_answer[-200:]}")
score_B, verdicts_B = faithfulness_decompose_verify(fabricated_answer, context_str)

# ── Cross-check with RAGAS's own faithfulness score ──────────────
ragas_score = df_adv.loc[df_adv["question"] == target_q, "faithfulness"].iloc[0] \
    if "faithfulness" in df_adv.columns and (df_adv["question"] == target_q).any() \
    else float("nan")

print("\n" + "═" * 74)
print(" 📊 FINAL COMPARISON")
print("═" * 74)
summary = pd.DataFrame([
    {"source": "Our Decompose+Verify — REAL answer",       "faithfulness": round(score_A, 2)},
    {"source": "Our Decompose+Verify — FABRICATED answer", "faithfulness": round(score_B, 2)},
    {"source": "RAGAS built-in faithfulness (real answer)", "faithfulness": round(ragas_score, 2)},
])
print(summary.to_string(index=False))

print("\n💡 Take-aways:")
print("  • CASE A ≈ RAGAS score → confirms RAGAS runs THIS SAME algorithm under the hood")
print("  • CASE B  < CASE A     → decompose+verify successfully caught the injected lie")
print("  • The mechanism is ordinary Groq LLM calls — you could wire it as a")
print("    runtime guardrail in ~50 lines. RAGAS just packages it neatly.")


---
## 🎓 Part 4 — Evaluate the RAG with **ARES** (Automated RAG Evaluation System)

**ARES** (Stanford, [paper](https://arxiv.org/abs/2311.09476), [GitHub](https://github.com/stanford-futuredata/ARES)) is the industrial-scale cousin of RAGAS.

Where RAGAS uses a **general-purpose LLM as a judge on every row** (great quality, expensive at scale), ARES **trains a small dedicated judge on synthetic data** and then combines it with a **tiny human-labeled set** to produce **statistically-guaranteed confidence intervals** — so you can evaluate millions of queries cheaply *and* know how sure you should be.

### How ARES works — the 3 stages

```
       ┌──────────────────────────────┐
       │ 1. SYNTHETIC DATA GENERATION │  Take your documents → GPT-4 (or FLAN-T5)
       │                              │  generates (query, doc, label) triples
       └──────────────┬───────────────┘
                      │
                      ▼
       ┌──────────────────────────────┐
       │ 2. TRAIN A LIGHTWEIGHT JUDGE │  Fine-tune DeBERTa-v3-large (~400M) on the
       │                              │  synthetic set — one binary classifier per
       │                              │  metric (context-relevance / faithfulness /
       │                              │  answer-relevance)
       └──────────────┬───────────────┘
                      │
                      ▼
       ┌──────────────────────────────┐
       │ 3. EVALUATE + PPI            │  Judge scores your full RAG output cheaply.
       │  (Prediction-Powered         │  Combine with ~150 human labels → tight
       │   Inference)                 │  confidence intervals on the true score.
       └──────────────────────────────┘
```

**PPI is the key idea** — you get the *cost* of a machine judge with the *statistical rigor* of human labels. See [Angelopoulos et al. 2023](https://arxiv.org/abs/2301.09633).

### ARES's 3 metrics

| Metric | Question |
|---|---|
| **Context Relevance** | Is the retrieved context relevant to the query? |
| **Answer Faithfulness** | Is the generated answer grounded in the retrieved context? |
| **Answer Relevance** | Does the answer address the query? |

These map ≈ 1-to-1 with three of RAGAS's four metrics (RAGAS adds `context_recall`).

### RAGAS vs ARES — decision guide

| | **RAGAS** | **ARES** |
|---|---|---|
| Setup effort | ⚡ minutes | 🏗️ hours + a GPU |
| Cost per 10 000 rows | $$$ (judge LLM per row) | $ (small classifier + PPI) |
| Human labels needed | 0 (works reference-free for most metrics) | ~150–300 for PPI validation |
| Reproducibility | Judge-LLM variance | Deterministic once judge is trained |
| Confidence intervals | ❌ | ✅ statistically guaranteed |
| CI/CD integration | Easy | Medium (needs model artefacts) |
| Metrics coverage | 9+ (incl. context recall, entities recall) | 3 core |
| Language coverage | Multilingual out-of-box | Limited (mostly English) |
| Best for | Prototypes · workshops · < 5 k questions | Production monitoring · ≥ 100 k questions |

### When to use ARES

✅ **You have 100k+ production queries per week** and RAGAS + GPT-4 judge would cost too much
✅ **You need statistical confidence intervals** (regulated domain, contract SLAs)
✅ **You have a small team of labellers** who can produce 150-300 gold labels
✅ **You have a GPU** for the one-off judge training

### When to stick with RAGAS

✅ You're prototyping / iterating
✅ Your test set is < 5 k questions
✅ You don't want to train + version a classifier
✅ You need domain-specific metrics (Aspect Critique, Entities Recall, Noise Sensitivity)

**Rule of thumb:** ship with RAGAS, upgrade to ARES only when eval cost or statistical rigor becomes the bottleneck.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 1 — Install ARES
# ──────────────────────────────────────────────────────────────────
# ⚠️  Heavy install: pulls torch, transformers, deberta-v3, scipy, ...
#     ~1.5 GB, ~2 min on Colab. Skip if you already have ARES.
# ══════════════════════════════════════════════════════════════════
!pip install -q ares-ai


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 2 — Prepare ARES-format TSVs from our RAG outputs
# ──────────────────────────────────────────────────────────────────
# ARES uses TSV files with columns:
#   Query · Document · Answer · Context_Relevance_Label ·
#          Answer_Faithfulness_Label · Answer_Relevance_Label
#
# For the workshop we reuse the same test set + pipeline outputs
# already produced above (`naive_rows`, `adv_rows`).
# ══════════════════════════════════════════════════════════════════
import os
import pandas as pd
from pathlib import Path

ARES_DIR = Path("ares_workspace"); ARES_DIR.mkdir(exist_ok=True)

def to_ares_tsv(rows: dict, path: Path) -> Path:
    """Flatten a `{question, answer, contexts, ground_truth}` dict → ARES TSV."""
    records = []
    for q, a, ctxs, gt in zip(
        rows["question"], rows["answer"], rows["contexts"], rows["ground_truth"]
    ):
        # ARES scores (query, single-document) pairs → one row per retrieved chunk
        for doc in ctxs:
            records.append({
                "Query": q,
                "Document": doc.replace("\n", " ").strip(),
                "Answer": a,
                # Optional gold labels for validation / few-shot examples
                "Context_Relevance_Label": "",
                "Answer_Faithfulness_Label": "",
                "Answer_Relevance_Label": "",
                "Ground_Truth": gt,
            })
    df = pd.DataFrame(records)
    df.to_csv(path, sep="\t", index=False)
    print(f"   ✅ {path}  ({len(df)} rows)")
    return path

print("▶ Exporting RAG outputs to ARES TSVs …")
naive_tsv = to_ares_tsv(naive_rows, ARES_DIR / "naive_rag_eval.tsv")
adv_tsv   = to_ares_tsv(adv_rows,   ARES_DIR / "advanced_rag_eval.tsv")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 3 — ARES workflow: synthetic queries → train judge → PPI eval
# ──────────────────────────────────────────────────────────────────
# ⚠️  A full ARES run needs:
#      • ~1 GB GPU RAM (DeBERTa-v3-large)
#      • an OpenAI or vLLM key for synthetic-data generation
#      • 10–30 minutes wall-clock
#
# The block below is a WIRED-UP SKELETON. It runs on CPU as a dry-run
# (falls back to a helpful message if ARES / GPU / OpenAI key are missing),
# so the workshop stays instant while showing the exact production API.
# ══════════════════════════════════════════════════════════════════
def run_ares_demo():
    try:
        from ares import ARES
    except ImportError:
        print("❌ `ares-ai` not installed — run STEP 1 first.")
        return None

    # ---------- 3a. Synthetic query generation ----------
    synth_config = {
        "document_filepaths":           [str(adv_tsv)],   # your corpus / RAG docs
        "few_shot_prompt_filename":     "few_shot_prompt_filename.tsv",   # user-supplied
        "synthetic_queries_filenames":  [str(ARES_DIR / "synth_queries.tsv")],
        "documents_sampled":            25,               # bump to 1000+ in prod
        "model_choice":                 "google/flan-t5-xxl",  # or "gpt-4o-mini"
    }

    # ---------- 3b. Judge classifier training ----------
    classifier_config = {
        "training_dataset":     [str(ARES_DIR / "synth_queries.tsv")],
        "validation_set":       [str(ARES_DIR / "gold_labels.tsv")],   # 150-300 human labels
        "label_column":         ["Context_Relevance_Label",
                                 "Answer_Faithfulness_Label",
                                 "Answer_Relevance_Label"],
        "num_epochs":           10,
        "patience_value":       3,
        "learning_rate":        5e-6,
        "assigned_batch_size":  8,
        "gradient_accumulation_multiplier": 4,
    }

    # ---------- 3c. Prediction-Powered Inference ----------
    ppi_config = {
        "evaluation_datasets":  [str(naive_tsv), str(adv_tsv)],
        "few_shot_examples_filepath": "few_shot_prompt_filename.tsv",
        "checkpoints":          ["./trained_ares_judge.pt"],
        "labels":               ["Context_Relevance_Label",
                                 "Answer_Faithfulness_Label",
                                 "Answer_Relevance_Label"],
        "gold_label_paths":     [str(ARES_DIR / "gold_labels.tsv")],
    }

    # ---------- Actually run — comment/uncomment as needed ----------
    print("▶ ARES config prepared — sample keys:")
    for k, v in list({**synth_config, **classifier_config, **ppi_config}.items())[:6]:
        print(f"   {k:<32}= {v}")

    # === Uncomment to actually execute (needs GPU + OpenAI key + human labels) ===
    # ares = ARES(synthetic_query_generator=synth_config)
    # ares.generate_synthetic_data()
    #
    # ares = ARES(classifier_model=classifier_config)
    # ares.train_classifier()
    #
    # ares = ARES(ppi=ppi_config)
    # results = ares.evaluate_RAG()
    # return results

    print("\n💡 Skeleton only — uncomment the ARES(...) calls above to run.")
    print("   Requires: GPU · 150-300 human labels · OpenAI/vLLM key.")
    return None


run_ares_demo()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 4 — ARES-STYLE REPORT (using RAGAS scores as a stand-in judge)
# ──────────────────────────────────────────────────────────────────
# ARES reports each metric as a POINT ESTIMATE + 95 % CONFIDENCE
# INTERVAL from PPI. Below we mimic that output shape so you know
# what a real ARES run will look like when you flip the switch above.
#
# Note: the CIs shown here are computed with a normal approximation on
# our tiny (n=4) RAGAS scores, so they will be WIDE. In production ARES
# gives tight CIs from thousands of judge scores + ~200 human labels.
# ══════════════════════════════════════════════════════════════════
import math
import statistics

Z_95 = 1.96  # 95 % normal-approx multiplier

# Map RAGAS metric names → ARES metric names
RAGAS_TO_ARES = {
    "context_precision": "Context Relevance",
    "faithfulness":      "Answer Faithfulness",
    "answer_relevancy":  "Answer Relevance",
}

def ares_style_ci(scores):
    """Point estimate + 95 % CI half-width (normal approximation)."""
    n = len(scores)
    if n == 0:
        return float("nan"), float("nan")
    mean = sum(scores) / n
    if n < 2:
        return mean, float("nan")
    sd = statistics.stdev(scores)
    half = Z_95 * sd / math.sqrt(n)
    return mean, half

print("=" * 45)
print("ARES-STYLE REPORT  (Mean ± 95 % CI half-width)")
print("=" * 45)

for pipe_label, df_pipe in [("Naive RAG", df_naive), ("Advanced RAG", df_adv)]:
    print(f"\n{pipe_label}\n")
    for ragas_col, ares_name in RAGAS_TO_ARES.items():
        if ragas_col not in df_pipe.columns:
            continue
        scores = df_pipe[ragas_col].dropna().tolist()
        mean, half = ares_style_ci(scores)
        if math.isnan(half):
            print(f"{ares_name:<20} : {mean:.2f}   (CI n/a — need ≥ 2 rows)")
        else:
            print(f"{ares_name:<20} : {mean:.2f}  ± {half:.2f}")

# Winner across the 3 ARES metrics
def mean_of(df_pipe):
    return sum(df_pipe[c].mean() for c in RAGAS_TO_ARES if c in df_pipe.columns) / \
           max(1, sum(c in df_pipe.columns for c in RAGAS_TO_ARES))

adv_avg   = mean_of(df_adv)
naive_avg = mean_of(df_naive)
winner = "Advanced RAG" if adv_avg > naive_avg else ("Naive RAG" if naive_avg > adv_avg else "Tie")

print("\nWinner (mean of 3 ARES metrics):")
print(f"  {winner}   [Advanced {adv_avg:.2f}  vs  Naive {naive_avg:.2f}]")
print("\n💡 To get REAL ARES scores with tight CIs, uncomment the ARES(...)")
print("   calls in STEP 3 and provide gold_labels.tsv (150-300 rows).")


### 📎 ARES production checklist

Before flipping the switch in STEP 3, gather:

| Artefact | Size | Notes |
|---|---|---|
| **Document corpus** | any | The docs your RAG retrieves from |
| **Few-shot prompt TSV** | ~5 rows | 5 human-written (query, doc, label) examples for the synth generator |
| **Human-labelled gold set** | 150-300 rows | Each row scored 0/1 on the 3 metrics; drives PPI's statistical validity |
| **GPU** | 8-16 GB VRAM | T4 works for training the DeBERTa judge |
| **Synth-gen LLM key** | 1 | `OPENAI_API_KEY` for GPT-4o-mini, or a local vLLM endpoint |
| **Trained judge checkpoint** | ~500 MB | Cache & version this — do NOT retrain per eval run |

### 🔗 Further reading
- [ARES paper — Automated Evaluation Framework for RAG](https://arxiv.org/abs/2311.09476)
- [ARES GitHub — quickstart, configs, examples](https://github.com/stanford-futuredata/ARES)
- [Prediction-Powered Inference — Angelopoulos et al., 2023](https://arxiv.org/abs/2301.09633)
- [Compare: RAGAS docs](https://docs.ragas.io) · [DeepEval](https://github.com/confident-ai/deepeval) · [TruLens](https://github.com/truera/trulens)
